In [1]:
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv("userbehaviour.csv")

In [5]:
df.head(2)

,userid,Average Screen Time,Average Spent on App (INR),Left Review,Ratings,New Password Request,Last Visited Minutes,Status
0,1001,17.0,634.0,1,9,7,2990,Installed
1,1002,0.0,54.0,0,4,8,24008,Uninstalled


In [9]:
df.isnull().sum()

userid                        0
Average Screen Time           0
Average Spent on App (INR)    0
Left Review                   0
Ratings                       0
New Password Request          0
Last Visited Minutes          0
Status                        0
dtype: int64

In [11]:
df["Status"].value_counts()

Status
Installed      916
Uninstalled     83
Name: count, dtype: int64

In [13]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df["Status"]=le.fit_transform(df["Status"])
df.head(3)

,userid,Average Screen Time,Average Spent on App (INR),Left Review,Ratings,New Password Request,Last Visited Minutes,Status
0,1001,17.0,634.0,1,9,7,2990,0
1,1002,0.0,54.0,0,4,8,24008,1
2,1003,37.0,207.0,0,8,5,971,0


In [15]:
x=df.drop(columns="Status")
y=df["Status"]

In [17]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [19]:
df["Status"].value_counts()

Status
0    916
1     83
Name: count, dtype: int64

In [21]:
from imblearn.under_sampling import RandomUnderSampler
randomsamp=RandomUnderSampler(random_state=42)
x_resampled,y_resampled=randomsamp.fit_resample(x_train,y_train)

In [23]:
# After resampling
from collections import Counter

print("Before Resampling:", Counter(y_train))
print("After Resampling:", Counter(y_resampled))


Before Resampling: Counter({0: 729, 1: 70})
After Resampling: Counter({0: 70, 1: 70})


In [25]:
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder , StandardScaler
from sklearn.metrics import accuracy_score

In [27]:
sc = StandardScaler()
x_train_scaled = sc.fit_transform(x_resampled)
x_test_scaled = sc.transform(x_test)

In [29]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Dense(units = 128 , activation = 'relu' ,
                          input_dim = x_train_scaled.shape[1]),  #input layer
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(units = 64 , activation = 'relu'),    #hidden layer
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(units = 1 , activation = 'sigmoid')   #output layer
])

C:\Users\shaba\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [31]:
loss_fn = tf.keras.losses.BinaryCrossentropy()
metrics = ['accuracy']

In [33]:
learning_rate = 0.001
momentum = 0.9
optimizer = tf.keras.optimizers.SGD(learning_rate = learning_rate ,
                                    momentum = momentum , nesterov = True)

In [35]:
model.compile(optimizer = optimizer , loss = loss_fn , metrics = metrics)

In [37]:
model.fit(x_train_scaled ,
         y_resampled ,
         epochs = 50 ,
          batch_size = 8 ,
          validation_split = 0.1

         )

Epoch 1/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 3s 38ms/step - accuracy: 0.5431 - loss: 0.6773 - val_accuracy: 0.0714 - val_loss: 0.8455
Epoch 2/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5843 - loss: 0.6535 - val_accuracy: 0.2143 - val_loss: 0.7523
Epoch 3/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7176 - loss: 0.5897 - val_accuracy: 0.7143 - val_loss: 0.6721
Epoch 4/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7996 - loss: 0.5541 - val_accuracy: 0.9286 - val_loss: 0.5999
Epoch 5/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8510 - loss: 0.5059 - val_accuracy: 0.9286 - val_loss: 0.5398
Epoch 6/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9318 - loss: 0.4516 - val_accuracy: 0.9286 - val_loss: 0.4914
Epoch 7/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9487 - loss: 0.4159 - val_accuracy: 0.9286 - val_loss: 0.4465
Epoch 8/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9060 - loss: 0.4156 - val_accuracy: 1.0000 - va

In [39]:
loss , accuracy = model.evaluate(x_test_scaled , y_test)

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9963 - loss: 0.0271


In [41]:
print("Total Accuracy :" , accuracy)

Total Accuracy : 0.9950000047683716
